In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsentregafinal")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = (
    f"abfss://{container}@{storageName}.dfs.core.windows.net/"
    "productos/productos_raw.xlsx"
)

print(ruta)

In [0]:
productos_schema = StructType([
    StructField("producto_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("marca", StringType(), True),
    StructField("costo", StringType(), True),
    StructField("precio", StringType(), True),
    StructField("activo", StringType(), True),
    StructField("fecha_alta", StringType(), True)
])

In [0]:
df_productos_read = (
    spark.read
    .format("dev.mauch.spark.excel")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("dataAddress", "'Sheet1'!A1")
    .schema(productos_schema)
    .load(ruta)
)

In [0]:
display(df_productos_read)

In [0]:
df_productos_ingestion_date = (
    df_productos_read
    .withColumn("ingestion_date", current_timestamp())
)

In [0]:
df_productos_final = df_productos_ingestion_date.select(
    "producto_id",
    "sku",
    "producto",
    "categoria",
    "marca",
    "costo",
    "precio",
    "activo",
    "fecha_alta",
    "ingestion_date"
)

In [0]:
df_productos_final.write.mode("overwrite").insertInto(
    f"{catalogo}.{esquema}.productos"
)

In [0]:
display(
    spark.table(f"{catalogo}.{esquema}.productos")
)

In [0]:
cantidad_origen = df_productos_read.count()

cantidad_bronze = spark.table(
    f"{catalogo}.{esquema}.productos"
).count()

print(f"Registros leídos del archivo: {cantidad_origen}")
print(f"Registros cargados en Bronze: {cantidad_bronze}")

if cantidad_origen != cantidad_bronze:
    raise ValueError(
        "La cantidad de registros del origen no coincide con Bronze."
    )

print("Carga de productos completada correctamente.")